### import libraries

In [1]:
import os
import yaml

import numpy as np
import matplotlib.pyplot as plt
from XLO_sim.XLO_sim import XLO_sim
from XLO_sim.Plot import XLO_plot
from XLO_sim import tools

import numpy.fft as fft

import scipy.constants as sp_const
au_in_eV = sp_const.value('atomic unit of energy') / sp_const.value('atomic unit of charge')

import multiprocessing as mp
import shutil

notebook_path = os.path.abspath("__file__")
notebook_directory = os.path.dirname(notebook_path)
base_directory = os.path.dirname(notebook_directory)

initializing ocelot...


### Transmission

In [ ]:
tpad = 1000
xpad = 64
ypad = 64

th_show = 2
w_show = 5

thx_show_min = -th_show
thx_show_max = th_show
thy_show_min = -th_show
thy_show_max = th_show
w_show_min = -w_show
w_show_max = w_show

In [3]:
def yaml_modify_seed_duration(input_yaml_path, output_yaml_path, new_seed_duration):
    # Read the original YAML file
    with open(input_yaml_path, 'r') as file:
        yaml_data = yaml.safe_load(file)
    
    # Modify the seed duration value
    yaml_data['seed_duration_FWHM_t'] = new_seed_duration
    
    new_tmax = max(yaml_data['tmax'], 4 * new_seed_duration)
    yaml_data['tmax'] = new_tmax
    
    # Write the modified data to a new YAML file
    with open(output_yaml_path, 'w') as file:
        yaml.safe_dump(yaml_data, file)
    
    print(f"Modified YAML file saved to {output_yaml_path} (tmax={new_tmax:.1f} fs)")

In [4]:
# Cu 2p core-hole lifetime is ~1 fs (Ka1 linewidth ~2.5 eV FWHM -> tau ~ hbar/Gamma ~0.3-1 fs),
# so this scan spans seed durations from well below (0.2 fs) to well above (60 fs) that
# timescale, to see the transition from a coherent, lifetime-dominated response to an
# incoherent average over many core-hole decay cycles.
ar_duration_values = [0.1, 0.2, 0.5, 1, 2, 3, 6]

ar_yaml = []

for duration in ar_duration_values:
    generated_directory = base_directory + '/config/generated'
    generated_path = os.path.join(generated_directory)
    if not os.path.exists(generated_path):
        os.makedirs(generated_path)
        
    input_yaml_path = base_directory + '/config/base/Cu-seed-SASE.yaml'
    output_yaml_path = base_directory + '/config/generated/Cu-seed-SASE_' + f'{duration:.2f}' + 'fs.yaml'
    yaml_modify_seed_duration(input_yaml_path, output_yaml_path, duration)
    ar_yaml.append(output_yaml_path)

Modified YAML file saved to /Users/parkinpham/Programming/Physics/DESY_Internship/Cu-RSA-XFEL-Simulation/config/generated/Cu-seed-SASE_0.10fs.yaml (tmax=8.0 fs)
Modified YAML file saved to /Users/parkinpham/Programming/Physics/DESY_Internship/Cu-RSA-XFEL-Simulation/config/generated/Cu-seed-SASE_0.20fs.yaml (tmax=8.0 fs)
Modified YAML file saved to /Users/parkinpham/Programming/Physics/DESY_Internship/Cu-RSA-XFEL-Simulation/config/generated/Cu-seed-SASE_0.50fs.yaml (tmax=8.0 fs)
Modified YAML file saved to /Users/parkinpham/Programming/Physics/DESY_Internship/Cu-RSA-XFEL-Simulation/config/generated/Cu-seed-SASE_1.00fs.yaml (tmax=8.0 fs)
Modified YAML file saved to /Users/parkinpham/Programming/Physics/DESY_Internship/Cu-RSA-XFEL-Simulation/config/generated/Cu-seed-SASE_2.00fs.yaml (tmax=8.0 fs)
Modified YAML file saved to /Users/parkinpham/Programming/Physics/DESY_Internship/Cu-RSA-XFEL-Simulation/config/generated/Cu-seed-SASE_3.00fs.yaml (tmax=12.0 fs)
Modified YAML file saved to /User

### calculating for given amount of repetitions for each yaml

In [ ]:
nrep = 100

# Get the number of available CPU cores
num_cpus = mp.cpu_count()

# Set the maximum number of processes to use
num_processes = num_cpus - 1  # Leave one core for system processes or other tasks
print("num_processes", num_processes)

   
data_path = os.path.join(base_directory, 'data/' + np.datetime_as_string(np.datetime64('now')))
if not os.path.exists(data_path):
    os.makedirs(data_path)

def run_simulation(yaml, run_path, rep):
    print('repetition ', rep + 1)

    X = XLO_sim(yaml)
    X.random_seed = rep  # Set the random seed for reproducibility
    seed_field = tools.Ocelot_SASE_seed_pstxy(X)
    X.configure(seed_field)
    X.run_3D()

    womega_ar, I_int_thy_w_0, I_thy0_w_0 = tools.SF_spectrum_w(X, 0, ypad, tpad)
    womega_ar, I_int_thy_w_last, I_thy0_w_last = tools.SF_spectrum_w(X, -1, ypad, tpad)

    date_string = np.datetime_as_string(np.datetime64('now'))
    np.savez_compressed(
        os.path.join(run_path, f"run_at_duration_{X.seed_duration_FWHM_t:.2f}_fs__repetition_{rep + 1}_{date_string}.npz"),
        womega_ar=womega_ar,
        I_int_thy_w_0=I_int_thy_w_0,
        I_thy0_w_0=I_thy0_w_0,
        I_int_thy_w_last=I_int_thy_w_last,
        I_thy0_w_last=I_thy0_w_last
    )

def run_repetitions_from_yaml(yaml, nrep, num_processes, data_path):
    date_string = np.datetime_as_string(np.datetime64('now'))
    X = XLO_sim(yaml)

    run_path = os.path.join(data_path, f'runs_duration_{X.seed_duration_FWHM_t:.2f}_fs_{date_string}')
    if not os.path.exists(run_path):
        os.makedirs(run_path)

    shutil.copy2(yaml, run_path)

    # Force the 'fork' start method explicitly so behavior is consistent
    # between macOS (default 'spawn' since Python 3.8) and the Linux
    # cluster (default 'fork'). 'spawn' fails here with AttributeError
    # because it re-imports __main__ to find run_simulation, which does
    # not work for functions defined in a notebook.
    ctx = mp.get_context('fork')
    with ctx.Pool(processes=num_processes) as pool:
        # Run simulations in parallel
        pool.starmap(run_simulation, [(yaml, run_path, rep) for rep in range(nrep)])

if __name__ == '__main__':
    for yaml_value in ar_yaml:
        print(f"Running simulations for YAML file: {yaml_value}")
        run_repetitions_from_yaml(yaml_value, nrep, num_processes, data_path)